# AuraGateway P0-P2 source input inspection V2

Metadata-only inspection. Attach exactly one successful materializer output, use Accelerator None and Internet Off.


In [ ]:
from __future__ import annotations

import hashlib
import json
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "ag-cu129-p0-p2-source-inspection-v2"
EXPECTED_DATASET_NAME = "ag-cu129-p0-p2-source-v2"
EXPECTED_OUTPUT_DIRECTORY = "ag_cu129_p0_p2_source_materializer_v2_output"
EXPECTED_SOURCE_BUNDLE_SHA256 = "8c90a0f294cd33a74b5e90da6b9f5671f2fab5bf1dcc0359f275664fce51f00c"
EXPECTED_BUNDLE_MANIFEST_SHA256 = "246937c7fe66460953d88ea05fce2a9244ea4f104793b54ab6a40b122cba4ede"
EXPECTED_SOURCE_INVENTORY_SHA256 = (
    "855b1e77900cd5e022255d12189fce4207bf93f74671fed9ec0d74caaf29d505"
)
EXPECTED_SOURCE_REPOSITORY_COMMIT = "831b4ad4e8eb4139b51af927eb721989be197cbc"
MATERIALIZATION_RECEIPT_NAME = "materialization_receipt.json"
SOURCE_INVENTORY_NAME = "source_inventory.json"
SHA256_MANIFEST_NAME = "sha256_manifest.json"
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()
REPORT_PATH = WORK_ROOT / "p0_p2_source_input_inspection_report.json"
EVIDENCE_ZIP = WORK_ROOT / "ag-cu129-p0-p2-source-inspection-v2.zip"


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_basename(value: str) -> str:
    path = PurePosixPath(value)
    if (
        path.is_absolute()
        or len(path.parts) != 1
        or ".." in path.parts
        or "\\" in value
        or value in {".", ".."}
    ):
        raise RuntimeError(f"unsafe materialized source path: {value}")
    return value


def discover_dataset() -> tuple[Path, dict[str, object]]:
    candidates: list[tuple[Path, dict[str, object]]] = []
    for receipt_path in INPUT_ROOT.rglob(MATERIALIZATION_RECEIPT_NAME):
        if not receipt_path.is_file() or receipt_path.is_symlink():
            continue
        dataset_root = receipt_path.parent.resolve()
        if INPUT_ROOT not in dataset_root.parents:
            continue
        try:
            raw = json.loads(receipt_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if not isinstance(raw, dict):
            continue
        receipt = {str(key): value for key, value in raw.items()}
        if (
            receipt.get("status") == "P0_P2_SOURCE_MATERIALIZED_V2"
            and receipt.get("output_dataset_name")
            == EXPECTED_DATASET_NAME
            and receipt.get("output_directory")
            == EXPECTED_OUTPUT_DIRECTORY
            and receipt.get("source_repository_commit")
            == EXPECTED_SOURCE_REPOSITORY_COMMIT
            and receipt.get("source_bundle_sha256")
            == EXPECTED_SOURCE_BUNDLE_SHA256
        ):
            candidates.append((dataset_root, receipt))
    if len(candidates) != 1:
        raise RuntimeError(
            "expected exactly one identity-shaped P0-P2 source dataset, "
            f"observed {len(candidates)}"
        )
    return candidates[0]


def validate_source_files(
    dataset_root: Path,
    inventory: list[object],
    sha_manifest: dict[str, object],
) -> tuple[str, int]:
    notebook_name: str | None = None
    source_count = 0
    for raw in inventory:
        if not isinstance(raw, dict):
            raise RuntimeError("source inventory entry is invalid")
        path_value = raw.get("path")
        sha_value = raw.get("sha256")
        size_value = raw.get("size_bytes")
        role_value = raw.get("role")
        if not isinstance(path_value, str):
            raise RuntimeError("source inventory path is invalid")
        if not isinstance(sha_value, str):
            raise RuntimeError("source inventory SHA-256 is invalid")
        if not isinstance(size_value, int):
            raise RuntimeError("source inventory size is invalid")
        name = safe_basename(path_value)
        path = dataset_root / name
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f"materialized source is missing: {name}")
        if path.stat().st_size != size_value:
            raise RuntimeError(f"materialized source size drifted: {name}")
        if sha256_file(path) != sha_value:
            raise RuntimeError(f"materialized source identity drifted: {name}")
        if sha_manifest.get(name) != sha_value:
            raise RuntimeError(f"SHA-256 manifest drifted: {name}")
        if role_value == "diagnostic_notebook":
            notebook_name = name
        source_count += 1
    if notebook_name is None:
        raise RuntimeError("diagnostic notebook is absent from inventory")
    if source_count != 3:
        raise RuntimeError("materialized source artifact count drifted")
    return notebook_name, source_count


def validate_notebook(path: Path) -> None:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(notebook, dict):
        raise RuntimeError("reviewed diagnostic notebook is invalid")
    cells = notebook.get("cells")
    if not isinstance(cells, list) or len(cells) != 2:
        raise RuntimeError("reviewed diagnostic notebook cell count drifted")
    for cell in cells:
        if not isinstance(cell, dict):
            raise RuntimeError("reviewed diagnostic notebook cell is invalid")
        if (
            cell.get("cell_type") == "code"
            and (
                cell.get("outputs") != []
                or cell.get("execution_count") is not None
            )
        ):
            raise RuntimeError(
                "reviewed diagnostic notebook contains execution state"
            )


def main() -> None:
    if REPORT_PATH.exists() or EVIDENCE_ZIP.exists():
        raise RuntimeError("inspection output path already exists")

    dataset_root, receipt = discover_dataset()
    inventory_path = dataset_root / SOURCE_INVENTORY_NAME
    sha_manifest_path = dataset_root / SHA256_MANIFEST_NAME
    inventory_bytes = inventory_path.read_bytes()
    if sha256_bytes(inventory_bytes) != EXPECTED_SOURCE_INVENTORY_SHA256:
        raise RuntimeError("mounted source inventory identity drifted")
    raw_inventory = json.loads(inventory_bytes.decode("utf-8"))
    raw_manifest = json.loads(sha_manifest_path.read_text(encoding="utf-8"))
    if not isinstance(raw_inventory, list):
        raise RuntimeError("mounted source inventory must be one array")
    if not isinstance(raw_manifest, dict):
        raise RuntimeError("mounted SHA-256 manifest must be one object")
    sha_manifest = {str(key): value for key, value in raw_manifest.items()}
    notebook_name, source_count = validate_source_files(
        dataset_root,
        raw_inventory,
        sha_manifest,
    )
    if sha_manifest.get(SOURCE_INVENTORY_NAME) != EXPECTED_SOURCE_INVENTORY_SHA256:
        raise RuntimeError("source inventory manifest binding drifted")
    if receipt.get("bundle_manifest_sha256") != EXPECTED_BUNDLE_MANIFEST_SHA256:
        raise RuntimeError("materialization bundle-manifest binding drifted")
    if receipt.get("source_inventory_sha256") != EXPECTED_SOURCE_INVENTORY_SHA256:
        raise RuntimeError("materialization inventory binding drifted")
    if receipt.get("source_file_count") != source_count:
        raise RuntimeError("materialization receipt file count drifted")
    zero_budget_fields = (
        "model_loads",
        "worker_starts",
        "model_requests",
        "benchmark_trajectory_requests",
        "external_spend",
    )
    if any(receipt.get(name) != 0 for name in zero_budget_fields):
        raise RuntimeError("materialization receipt execution budget drifted")
    if receipt.get("credentials_present") is not False:
        raise RuntimeError("materialization receipt credential state drifted")
    if receipt.get("customer_data_present") is not False:
        raise RuntimeError("materialization receipt customer-data state drifted")

    validate_notebook(dataset_root / notebook_name)
    report = {
        "schema_version": "2.0.0",
        "status": "P0_P2_SOURCE_INPUT_INSPECTION_PASSED_V2",
        "inspection_notebook_name": NOTEBOOK_NAME,
        "source_repository_commit": EXPECTED_SOURCE_REPOSITORY_COMMIT,
        "dataset_root": str(dataset_root),
        "output_dataset_name": EXPECTED_DATASET_NAME,
        "source_bundle_sha256": EXPECTED_SOURCE_BUNDLE_SHA256,
        "bundle_manifest_sha256": EXPECTED_BUNDLE_MANIFEST_SHA256,
        "source_inventory_sha256": EXPECTED_SOURCE_INVENTORY_SHA256,
        "source_file_count": source_count,
        "notebook_outputs_present": False,
        "notebook_execution_counts_present": False,
        "network_requests": 0,
        "credentials_used": False,
        "customer_data_present": False,
        "model_loads": 0,
        "worker_starts": 0,
        "model_requests": 0,
        "benchmark_trajectory_requests": 0,
        "external_spend": 0,
        "next_gate": (
            "integrate_materialized_p0_p2_source_with_execution_launcher_v2"
        ),
    }
    REPORT_PATH.write_text(canonical_json(report), encoding="utf-8")
    with zipfile.ZipFile(
        EVIDENCE_ZIP,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        archive.write(REPORT_PATH, arcname=REPORT_PATH.name)
        archive.write(
            dataset_root / MATERIALIZATION_RECEIPT_NAME,
            arcname=MATERIALIZATION_RECEIPT_NAME,
        )
        archive.write(
            dataset_root / SOURCE_INVENTORY_NAME,
            arcname=SOURCE_INVENTORY_NAME,
        )
        archive.write(
            dataset_root / SHA256_MANIFEST_NAME,
            arcname=SHA256_MANIFEST_NAME,
        )
    print(
        canonical_json(
            {
                **report,
                "inspection_evidence_zip": str(EVIDENCE_ZIP),
                "inspection_evidence_zip_sha256": sha256_file(
                    EVIDENCE_ZIP
                ),
            }
        )
    )


main()